# 04 — Arm 3: Frozen retrieval shortlist + LLM selection

Dataset C, AMLH coursework. Frozen Arm 1 retrieval produces a 20-label shortlist; a seq2seq LLM then selects one label under one of three prompt conditions. `answer` is never an input at inference time. All substantive logic lives in `src/amlh/arm3_llm.py`; this notebook only loads artefacts, prints prompt-budget diagnostics, runs the condition loop, and persists tables to `artefacts/` and figures to `figures/`. It runs standalone after a kernel restart by loading `artefacts/split_fit.csv` and `artefacts/split_val.csv`.

In [ ]:
import time

import pandas as pd
import torch
from transformers import AutoTokenizer

from amlh import arm3_llm as a3
from amlh.config import ARTEFACTS_DIR, FIGURES_DIR, HYPERPARAMETERS, SEED, set_seed

## 1. Load splits from `artefacts/`

In [ ]:
set_seed()

fit = pd.read_csv(ARTEFACTS_DIR / "split_fit.csv")
val = pd.read_csv(ARTEFACTS_DIR / "split_val.csv")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print(f"fit={len(fit)} ({fit.disease.nunique()} classes) | val={len(val)} ({val.disease.nunique()} classes)")
print(f"device={device} | gpu={gpu_name}")

shortlist_k = HYPERPARAMETERS.shortlist_k
llm_temperature = HYPERPARAMETERS.llm_temperature
arm3_model_name = a3.selected_model_name(HYPERPARAMETERS)
n_shots = 2
prompt_modes = ["zero_shot", "few_shot", "cot"]
print({"shortlist_k": shortlist_k, "llm_temperature": llm_temperature, "prompt_mode": None, "n_shots": n_shots, "arm3_model_name": arm3_model_name})

## 2. Prompt budget check

Build the prompts first, then measure their token lengths and the truncation rate at 512 before any condition is run. Labels are prettified with underscores replaced by spaces, and candidates are numbered so the model only has to emit an index.

In [ ]:
shortlist_rankings, top_sim = a3.build_shortlist_ranking(fit, val, HYPERPARAMETERS, depth=shortlist_k)
examples = a3.build_examples(fit, n=n_shots, seed=SEED)
tokeniser = AutoTokenizer.from_pretrained(arm3_model_name)

prompts_by_mode = {}
budget_rows = []
for mode in prompt_modes:
    prompts = a3.build_prompts_for_condition(
        val, shortlist_rankings, mode, examples=examples if mode == "few_shot" else None
    )
    prompts_by_mode[mode] = prompts
    summary = a3.prompt_token_lengths(prompts, tokeniser, max_length=512)
    budget_rows.append({
        "condition": mode,
        "min_tokens": summary["min"],
        "median_tokens": summary["median"],
        "mean_tokens": summary["mean"],
        "p95_tokens": summary["p95"],
        "max_tokens": summary["max"],
        "truncation_rate_512": summary["truncation_rate"],
    })

budget_df = pd.DataFrame(budget_rows)
print(budget_df.to_string(index=False))

if budget_df["truncation_rate_512"].max() > 0:
    print("WARNING: at least one prompt is truncated at 512 tokens. Inspect before shortening anything.")
else:
    print("No prompt truncation at 512 tokens.")

## 3. Run all prompt conditions

Load Flan-T5-large in float32, then run the three prompt conditions on the standard validation hold-out. The generator is greedy (`do_sample=False`, temperature 0).

In [ ]:
model_tokeniser, model = a3.load_generator(arm3_model_name, device=device)

condition_frames = {}
condition_rows = []
condition_prompts = {}

for mode in prompt_modes:
    start = time.perf_counter()
    pred_df, metrics, prompts = a3.run_condition(
        fit,
        val,
        HYPERPARAMETERS,
        mode=mode,
        tokeniser=model_tokeniser,
        model=model,
        device=device,
        examples=examples if mode == "few_shot" else None,
        shortlist_depth=shortlist_k,
    )
    elapsed = time.perf_counter() - start
    metrics["wall_clock_sec"] = elapsed
    condition_frames[mode] = pred_df
    condition_rows.append(metrics)
    condition_prompts[mode] = prompts

condition_df = pd.DataFrame(condition_rows)
print(condition_df.to_string(index=False))

## 4. Pairwise McNemar and prompt-mode selection

If no condition separates from the others at p < 0.05, the comparison is unresolved and `zero_shot` is kept on the declared prior.

In [ ]:
mcnemar_df = a3.pairwise_condition_mcnemar(condition_frames)
print(mcnemar_df.to_string(index=False))

selected_mode, tie_break_fired = a3.select_prompt_mode(condition_df, mcnemar_df)
print({"selected_mode": selected_mode, "tie_break_fired": tie_break_fired})

predictions_df = pd.concat(condition_frames.values(), ignore_index=True)
predictions_df.to_csv(ARTEFACTS_DIR / "arm3_val_predictions.csv", index=False)
condition_df.to_csv(ARTEFACTS_DIR / "arm3_prompt_conditions.csv", index=False)
mcnemar_df.to_csv(ARTEFACTS_DIR / "arm3_condition_mcnemar.csv", index=False)
with open(ARTEFACTS_DIR / "arm3_prompts.txt", "w", encoding="utf-8") as f:
    f.write(a3.build_prompt_table(condition_prompts))

print("frozen config values to copy into config.py:")
print({"shortlist_k": shortlist_k, "llm_temperature": llm_temperature, "prompt_mode": selected_mode, "n_shots": n_shots, "arm3_model_name": arm3_model_name})